# Categorical Generalization of PII

Applies categorical generalization to the `job_title` column of `pii_demo.pii_source.employees`: specific job titles are replaced with broader categories (Engineering, Data & Analytics, Management, Sales & Business), reducing identifiability while preserving analytical value. The result is written to `pii_demo.pii_target.employees_categorical_generalized`.

In [0]:
%python
from pyspark.sql import functions as F

# Read the source table
df = spark.table("pii_demo.pii_source.employees")

# Define categorical generalization mapping for job titles
# Specific job titles will be grouped into broader categories
job_category_mapping = {
    "Software Engineer": "Engineering",
    "Data Scientist": "Data & Analytics",
    "Product Manager": "Management",
    "Sales Executive": "Sales & Business",
    "HR Manager": "Management"
}

# Create a mapping expression using CASE WHEN
mapping_expr = F.when(F.col("job_title") == "Software Engineer", "Engineering") \
    .when(F.col("job_title") == "Data Scientist", "Data & Analytics") \
    .when(F.col("job_title") == "Product Manager", "Management") \
    .when(F.col("job_title") == "Sales Executive", "Sales & Business") \
    .when(F.col("job_title") == "HR Manager", "Management") \
    .otherwise("Other")

In [0]:
%python
# Apply categorical generalization to job_title
df_generalized = df.withColumn("job_title_generalized", mapping_expr) \
    .drop("job_title") \
    .withColumnRenamed("job_title_generalized", "job_title")

# Write to target table
df_generalized.write.mode("overwrite").saveAsTable("pii_demo.pii_target.employees_categorical_generalized")

print("Categorical generalization applied successfully!")
print(f"Total records processed: {df_generalized.count()}")

In [0]:
%python
# Show a sample of generalized data
display(df_generalized.limit(10))